# Analysis of Emergency Obstetric Care (EmOC) in Pasto, Colombia
> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../pereira/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate isochrones on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r path/to/requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [2]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

cannot find .env file


ValueError: No API key was specified. Please visit https://openrouteservice.org/sign-up to create one.

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [3]:
# Set paths to access Kano data
# Define directories
data_inputs = '../scripts/Pasto/data-inputs/'
data_temp = '../scripts/Pasto/data-temp/'
model_outputs = '../Pasto/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined based on the project's researchers and their local context, based on data obtained from the [datasets of health facilities](https://www.datos.gov.co/Salud-y-Protecci-n-Social/Registro-Especial-de-Prestadores-y-Sedes-de-Servic/c36g-9fc2/about_data).

In [3]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities.geojson')

In [4]:
healthcare_facilities_validated = healthcare_facilities_validated[healthcare_facilities_validated['project_validation'] != 'No EmOC']
healthcare_facilities_validated

,CodigoPrestador,primary_hc,basic_emoc,comp_emoc,NombrePrestador,CodigoHabilitacionSede,NombreSede,TipoIdentificacion,NumeroIdentificacion,NaturalezaJuridica,...,EmailSede,TelefonoSede,ClasePrestadorDesc,FechaCorte,address_geocoding,status,latitude,longitude,project_validation,geometry
33,5200100096,None,x,x,HOSPITAL SAN RAFAEL DE PASTO,5.200100e+11,HOSPITAL SAN RAFAEL DE PASTO,NI,891200274,Privada,...,hsrpasto@hospitalsanrafaelpasto.com,7362680,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"CALLE 15 No 42C-35, PASTO, Narino, COLOMBIA",OK,1.220111,-77.287025,Private Comprehensive EmOC,POINT (-77.28702 1.22011)
39,5200100121,?,?,None,FUNDACION MARIA FORTALEZA,5.200100e+11,FUNDACION MARIA FORTALEZA,NI,814000463,Privada,...,mariafortaleza@hotmail.com,7290904,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"Carrera 38 NO 19 - 41, PASTO, Narino, COLOMBIA",OK,1.226692,-77.284158,Private Basic EmOC,POINT (-77.28416 1.22669)
72,5200100261,None,x,x,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,5.200100e+11,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,NI,891200372,Privada,...,diripsnarino@cruzrojacolombiana.org,7292886,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"Carrera 25 No. 13-26, PASTO, Narino, COLOMBIA",OK,1.213070,-77.282908,Private Comprehensive EmOC,POINT (-77.28291 1.21307)
76,5200100279,None,x,None,CLINICA NUESTRA SEÑORA DE FATIMA S.A.,5.200100e+11,CLINICA NUESTRA SEÑORA DE FATIMA,NI,891200032,Privada,...,gerencia@clifatima.com,7333630 EXTENCION 289,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"CALLE 21 NUMERO 26 40, PASTO, Narino, COLOMBIA",OK,1.217334,-77.276523,Private Basic EmOC,POINT (-77.27652 1.21733)
77,5200100283,None,x,None,HOSPITAL INFANTIL LOS ANGELES,5.200100e+11,HOSPITAL INFANTIL LOS ANGELES,NI,891200240,Privada,...,asesorcalidad@correohila.org,7311533,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"KR 32 # 21 A 30, PASTO, Narino, COLOMBIA",OK,1.222850,-77.280222,Private Basic EmOC,POINT (-77.28022 1.22285)
130,5200100557,x,x,None,FUNDACION HOSPITAL SAN PEDRO,5.200100e+11,FUNDACION HOSPITAL SAN PEDRO,NI,891200209,Privada,...,fhsp@hospitalsanpedro.org,7336000,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"CL16 KR43 ESQ, PASTO, Narino, COLOMBIA",OK,1.224253,-77.290468,Private Basic EmOC,POINT (-77.29047 1.22425)
240,5200101102,x,x,x,E.S.E. HOSPITAL UNIVERSITARIO DEPARTAMENTAL DE...,5.200100e+11,HOSPITAL UNIVERSITARIO DEPARTAMENTAL DE NARIÑO,NI,891200528,Pública,...,hudn@hosdenar.gov.co,3113640760,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"Calle 22No. 7-93, PASTO, Narino, COLOMBIA",OK,1.204546,-77.261302,Public Comprehensive EmOC,POINT (-77.2613 1.20455)
242,5200101114,None,x,x,HNAS. HOSPITALARIAS DEL SAGDO. CORAZON DE JESU...,5.200100e+11,HNAS. HOSPITALARIAS DEL SAGDO. CORAZON DE JESU...,NI,860007760,Privada,...,direccioncientifica@hospitalperpetuosocorro.org,7235684 - 7235685,Instituciones Prestad oras de Servic ios de S ...,"KR 33 # 5 OESTE 104, PASTO, Narino, COLOMBIA",None,INVALID_REQUEST,NaN,NaN,Private Comprehensive EmOC,None
312,5200101457,None,x,None,EMPRESA SOCIAL DEL ESTADO PASTO SALUD E.S.E.,5.200100e+11,HOSPITAL LA ROSA,NI,900091143,Pública,...,dosur@pastosaludese.gov.co,7207183,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"DIAGONAL 12 A No. 3A-19, PASTO, Narino, COLOMBIA",OK,1.219852,-77.291324,Public Basic EmOC,POINT (-77.29132 1.21985)
314,5200101457,None,x,None,EMPRESA SOCIAL DEL ESTADO PASTO SALUD E.S.E.,5.200100e+11,HOSPITAL SANTA MONICA,NI,900091143,Pública,...,dooriente@pastosaludese.gov.co,CALLE 21 No. 9 ESTE A 88,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"CALLE 21 No. 9 ESTE A 88, PASTO, Narino, COLOMBIA",OK,1.201617,-77.262263,Public Basic EmOC

In [5]:
# Filtered out facilities that do not provide EmOC services
# to a new geo_json file
healthcare_facilities_validated.to_file(data_temp + 'healthcare_facilities_emoc.geojson', driver='GeoJSON')


### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [6]:
study_area = gpd.read_file(data_inputs + '100mGrid.gpkg')
raster_path = data_inputs + 'col_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [15]:
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.union_all().__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

In [16]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [27]:
with rasterio.open(data_inputs + 'pasto_col_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

Calculating the centroids for grid cells

In [19]:
rows, cols = np.where(band1 > 0)
grid_cells = [clipped_transform * (col + 0.5, row + 0.5) for row, col in zip(rows, cols)]
population_values = band1[rows, cols]

In [20]:
grid_df = pd.DataFrame(grid_cells, columns=["longitude", "latitude"])
grid_df["population"] = population_values

grid_df["rowid"] = range(1, len(grid_df) + 1)
population_centroids_gdf = gpd.GeoDataFrame(grid_df, geometry=[Point(xy) for xy in zip(grid_df["longitude"], grid_df["latitude"])])
population_centroids_gdf.set_crs("EPSG:4326", inplace=True)

population_centroids_gdf.to_file(data_temp + "population_centroids.gpkg", driver="GPKG")

In [21]:
population_centroids_gdf

,longitude,latitude,population,rowid,geometry
0,-77.312916,1.242917,221.282761,1,POINT (-77.31292 1.24292)
1,-77.304583,1.242917,369.811707,2,POINT (-77.30458 1.24292)
2,-77.296250,1.242917,407.195709,3,POINT (-77.29625 1.24292)
3,-77.287916,1.242917,328.971130,4,POINT (-77.28792 1.24292)
4,-77.279583,1.242917,285.810059,5,POINT (-77.27958 1.24292)
...,...,...,...,...,...
113,-77.312916,1.151250,138.998276,114,POINT (-77.31292 1.15125)
114,-77.304583,1.151250,254.474304,115,POINT (-77.30458 1.15125)
115,-77.296250,1.151250,160.856277,116,POINT (-77.29625 1.15125)
116,-77.287916,1.151250,124.766411,117,POINT (-77.28792 1.15125)


### Adding population data at 1km grid to 100m grid

In [22]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

# Use the respective EPSG code for Colombia UTM Zone 18N
epsg = 'EPSG:21818'

In [24]:
# Preparing grid
grid_file = data_inputs + '100mGrid.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'rowid', 'geometry', 'latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
grid

,grid_id,rowid,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,1,"POLYGON ((241672.347 138012.687, 241670.796 13...",1.245136,1.244731,1.245540,-77.318557,-77.319064,-77.318051
1,1,2,"POLYGON ((241783.457 138012.589, 241781.906 13...",1.245136,1.244731,1.245540,-77.317559,-77.318066,-77.317053
2,2,3,"POLYGON ((241894.566 138012.491, 241893.016 13...",1.245136,1.244731,1.245540,-77.316561,-77.317068,-77.316055
3,3,4,"POLYGON ((242005.676 138012.394, 242004.125 13...",1.245136,1.244731,1.245540,-77.315564,-77.316070,-77.315057
4,4,5,"POLYGON ((242116.785 138012.296, 242115.235 13...",1.245136,1.244731,1.245540,-77.314566,-77.315072,-77.314059
...,...,...,...,...,...,...,...,...,...
9995,9995,9996,"POLYGON ((245738.506 127361.863, 245737.075 12...",1.148884,1.148480,1.149289,-77.281957,-77.282462,-77.281451
9996,9996,9997,"POLYGON ((245849.614 127361.774, 245848.183 12...",1.148884,1.148480,1.149289,-77.280959,-77.281464,-77.280453
9997,9997,9998,"POLYGON ((245960.722 127361.685, 245959.291 12...",1.148884,1.148480,1.149289,-77.279961,-77.280467,-77.279455
9998,9998,9999,"POLYGON ((246071.831 127361.596, 246070.399 12...",1.148884,1.148480,1.149289,-77.278963,-77.279469,-77.278457


Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.


Downoad the building footprints data from Overture Maps. It uses the bounding box of the study area to limit the download siz (See the bounding box coordinates below). The dataset is natively provided as geoparket. We use the theme = buildings to filter only building footprints. This action is carried out using the Overturemaps python CLI. Therefore, run the following command in your terminal:

```bash
overturemaps download --bbox=-77.3190765,1.1484797,-77.2082901,1.2455403 -f geojson --type=building -o ../scripts/Pasto/data-temp/Pasto_GOBv3.geojson
```
**Note**: Make sure your terminal is pointed to the correct path where the notebook is located before running the command above. You might need to navigate as follows 
```bash
cd models/emergency-maternal-care/scripts/
overturemaps download --bbox=77.3190765,1.1484797,-77.2082901,1.2455403 -f geojson --type=building -o ../scripts/Pasto/data-temp/Pasto_GOBv3.geojson
```

In [25]:
# Count buildings per grid cell

# Loading Google building footprints
building_file = data_temp + 'Pasto_GOBv3.geojson'
buildings = gpd.read_file(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

# Joining buildings to grid
grid_buildings = grid.sjoin(buildings.set_geometry('centroid').drop(columns='geometry'), how='inner', predicate='intersects')
grid_buildings = grid_buildings.groupby('grid_id')

# Counting buildings per grid
building_counts = grid_buildings.size().rename('bcount')

# Adding building count to grid cells
grid = grid.merge(building_counts, on='grid_id', how='left')

# Assign building count 0 to cells with no buildings (NaN)
grid['bcount'] = grid['bcount'].fillna(0)
grid

,grid_id,rowid,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max,bcount
0,0,1,"POLYGON ((241672.347 138012.687, 241670.796 13...",1.245136,1.244731,1.245540,-77.318557,-77.319064,-77.318051,3.0
1,1,2,"POLYGON ((241783.457 138012.589, 241781.906 13...",1.245136,1.244731,1.245540,-77.317559,-77.318066,-77.317053,0.0
2,2,3,"POLYGON ((241894.566 138012.491, 241893.016 13...",1.245136,1.244731,1.245540,-77.316561,-77.317068,-77.316055,0.0
3,3,4,"POLYGON ((242005.676 138012.394, 242004.125 13...",1.245136,1.244731,1.245540,-77.315564,-77.316070,-77.315057,0.0
4,4,5,"POLYGON ((242116.785 138012.296, 242115.235 13...",1.245136,1.244731,1.245540,-77.314566,-77.315072,-77.314059,0.0
...,...,...,...,...,...,...,...,...,...,...
9995,9995,9996,"POLYGON ((245738.506 127361.863, 245737.075 12...",1.148884,1.148480,1.149289,-77.281957,-77.282462,-77.281451,0.0
9996,9996,9997,"POLYGON ((245849.614 127361.774, 245848.183 12...",1.148884,1.148480,1.149289,-77.280959,-77.281464,-77.280453,0.0
9997,9997,9998,"POLYGON ((245960.722 127361.685, 245959.291 12...",1.148884,1.148480,1.149289,-77.279961,-77.280467,-77.279455,1.0
9998,9998,9999,"POLYGON ((246071.831 127361.596, 246070.399 12...",1.148884,1.148480,1.149289,-77.278963,-77.279469,-77.278457,0.0


The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [28]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Loading coarse pop data
pop_file = data_path / 'pasto_col_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Converting the raster grid to vector data
pop_grid = raster2vector(pop_raster, transform, crs)
pop_grid = pop_grid.to_crs(epsg)
pop_grid['pop_grid_id'] = range(len(pop_grid))
# pop_grid.to_parquet(data_path / 'sanity_check_pop.parquet')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry','rowid', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

Index(['grid_id', 'rowid', 'geometry', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max', 'bcount', 'centroid', 'index_right',
       'pop_grid_pop', 'pop_grid_id'],
      dtype='object')


,grid_id,bcount,pop_grid_id,geometry,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,3.0,0,"POLYGON ((241672.347 138012.687, 241670.796 13...",1,1.245136,1.244731,1.24554,-77.318557,-77.319064,-77.318051
1,1,0.0,0,"POLYGON ((241783.457 138012.589, 241781.906 13...",2,1.245136,1.244731,1.24554,-77.317559,-77.318066,-77.317053
2,2,0.0,1,"POLYGON ((241894.566 138012.491, 241893.016 13...",3,1.245136,1.244731,1.24554,-77.316561,-77.317068,-77.316055
3,3,0.0,1,"POLYGON ((242005.676 138012.394, 242004.125 13...",4,1.245136,1.244731,1.24554,-77.315564,-77.316070,-77.315057
4,4,0.0,1,"POLYGON ((242116.785 138012.296, 242115.235 13...",5,1.245136,1.244731,1.24554,-77.314566,-77.315072,-77.314059


In [29]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

,grid_id,bcount,pop_grid_id,geometry_x,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,geometry_y,pop
0,0,3.0,0,"POLYGON ((241672.347 138012.687, 241670.796 13...",1,1.245136,1.244731,1.24554,-77.318557,-77.319064,-77.318051,28.0,0.107143,NaN,"POLYGON ((240852.514 138273.619, 241780.357 13...",NaN
1,1,0.0,0,"POLYGON ((241783.457 138012.589, 241781.906 13...",2,1.245136,1.244731,1.24554,-77.317559,-77.318066,-77.317053,28.0,0.000000,NaN,"POLYGON ((240852.514 138273.619, 241780.357 13...",NaN
2,2,0.0,1,"POLYGON ((241894.566 138012.491, 241893.016 13...",3,1.245136,1.244731,1.24554,-77.316561,-77.317068,-77.316055,130.0,0.000000,221.282761,"POLYGON ((241780.357 138272.798, 242708.195 13...",0.0
3,3,0.0,1,"POLYGON ((242005.676 138012.394, 242004.125 13...",4,1.245136,1.244731,1.24554,-77.315564,-77.316070,-77.315057,130.0,0.000000,221.282761,"POLYGON ((241780.357 138272.798, 242708.195 13...",0.0
4,4,0.0,1,"POLYGON ((242116.785 138012.296, 242115.235 13...",5,1.245136,1.244731,1.24554,-77.314566,-77.315072,-77.314059,130.0,0.000000,221.282761,"POLYGON ((241780.357 138272.798, 242708.195 13...",0.0


In [30]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])
grid.head()


,grid_id,bcount,pop_grid_id,geometry_x,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop
0,0,3.0,0,"POLYGON ((241672.347 138012.687, 241670.796 13...",1,1.245136,1.244731,1.24554,-77.318557,-77.319064,-77.318051,28.0,0.107143,NaN,NaN
1,1,0.0,0,"POLYGON ((241783.457 138012.589, 241781.906 13...",2,1.245136,1.244731,1.24554,-77.317559,-77.318066,-77.317053,28.0,0.000000,NaN,NaN
2,2,0.0,1,"POLYGON ((241894.566 138012.491, 241893.016 13...",3,1.245136,1.244731,1.24554,-77.316561,-77.317068,-77.316055,130.0,0.000000,221.282761,0.0
3,3,0.0,1,"POLYGON ((242005.676 138012.394, 242004.125 13...",4,1.245136,1.244731,1.24554,-77.315564,-77.316070,-77.315057,130.0,0.000000,221.282761,0.0
4,4,0.0,1,"POLYGON ((242116.785 138012.296, 242115.235 13...",5,1.245136,1.244731,1.24554,-77.314566,-77.315072,-77.314059,130.0,0.000000,221.282761,0.0


In [31]:
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-pasto-centroids.gpkg', driver='GPKG')

In [32]:
# if building data is of relevance. centroid gemetry to be deleted
buildings_footprint = buildings.drop(columns=['centroid'])
buildings_footprint.to_crs(4326)
buildings_footprint.to_file(data_temp + 'buildings-pasto.gpkg', driver='GPKG')

## 2. Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [ ]:
origin_gdf = population_centroids_gdf
origin_name_column = 'grid_code'
destination_gdf = healthcare_facilities_validated.dropna(subset=['geometry'])
destination_name_column = 'facility_name'

In [ ]:
origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))

In [ ]:
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))

In [ ]:
locations = origins + destinations

In [ ]:
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

In [ ]:
body = {'locations': locations,
       'destinations': destinations_index,
       'sources': origins_index,
       'metrics': ['distance', 'duration']}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

response = requests.post('https://api.openrouteservice.org/v2/matrix/driving-car', json=body, headers=headers)

In [ ]:
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities_validated[(destination_gdf.geometry.x == dest_x) & (destination_gdf.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# Convert the results into a DataFrame
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

In [ ]:
# Save to CSV
merged_df = pd.merge(matrix_df, grid_df[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
merged_df

In [ ]:
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [7]:
# If not loaded yet, read from the temporary folder
centroids_df = gpd.read_file(data_temp +'pop-grid-pasto-centroids.gpkg')
centroids_df

,grid_id,bcount,pop_grid_id,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop,geometry
0,0,3.0,0,1,1.245136,1.244731,1.245540,-77.318557,-77.319064,-77.318051,28.0,0.107143,NaN,NaN,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554..."
1,1,0.0,0,2,1.245136,1.244731,1.245540,-77.317559,-77.318066,-77.317053,28.0,0.000000,NaN,NaN,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554..."
2,2,0.0,1,3,1.245136,1.244731,1.245540,-77.316561,-77.317068,-77.316055,130.0,0.000000,221.282761,0.000000,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554..."
3,3,0.0,1,4,1.245136,1.244731,1.245540,-77.315564,-77.316070,-77.315057,130.0,0.000000,221.282761,0.000000,"POLYGON ((-77.31506 1.24473, -77.31507 1.24554..."
4,4,0.0,1,5,1.245136,1.244731,1.245540,-77.314566,-77.315072,-77.314059,130.0,0.000000,221.282761,0.000000,"POLYGON ((-77.31406 1.24473, -77.31407 1.24554..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,0.0,170,9996,1.148884,1.148480,1.149289,-77.281957,-77.282462,-77.281451,6.0,0.000000,94.708580,0.000000,"POLYGON ((-77.28145 1.14848, -77.28146 1.14929..."
9996,9996,0.0,170,9997,1.148884,1.148480,1.149289,-77.280959,-77.281464,-77.280453,6.0,0.000000,94.708580,0.000000,"POLYGON ((-77.28045 1.14848, -77.28047 1.14929..."
9997,9997,1.0,170,9998,1.148884,1.148480,1.149289,-77.279961,-77.280467,-77.279455,6.0,0.166667,94.708580,15.784763,"POLYGON ((-77.27946 1.14848, -77.27947 1.14929..."
9998,9998,0.0,170,9999,1.148884,1.148480,1.149289,-77.278963,-77.279469,-77.278457,6.0,0.000000,94.708580,0.000000,"POLYGON ((-77.27846 1.14848, -77.27847 1.14929..."


In [8]:
# If not loaded yet, read from the temporary folder
matrix_df = pd.read_csv(data_temp +'pasto_access.csv')
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,0,518.30,7.24
1,1,1,521.19,7.27
2,1,2,522.39,7.28
3,1,3,425.31,5.70
4,1,4,425.31,5.70
...,...,...,...,...
29995,3,9995,1191.14,11.47
29996,3,9996,1283.66,12.42
29997,3,9997,1286.86,12.44
29998,3,9998,1287.95,12.45


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the 2SFCA without a travel time estimate.

In [9]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,0,518.30,7.24
1,1,1,521.19,7.27
2,1,2,522.39,7.28
3,1,3,425.31,5.70
4,1,4,425.31,5.70
...,...,...,...,...
29995,3,9995,1191.14,11.47
29996,3,9996,1283.66,12.42
29997,3,9997,1286.86,12.44
29998,3,9998,1287.95,12.45


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [10]:
pop_centroids_hcf = pd.merge(matrix_df, centroids_df[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','bcount','pop_grid_bcount', 'pop_grid_pop', 'pop', 'geometry']], 
                     left_on='destination_id', right_on='grid_id', how='left')
pop_centroids_hcf

,origin_id,destination_id,duration_seconds,distance_km,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,bcount,pop_grid_bcount,pop_grid_pop,pop,geometry
0,1,0,518.30,7.24,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,3.0,28.0,NaN,NaN,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554..."
1,1,1,521.19,7.27,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,0.0,28.0,NaN,NaN,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554..."
2,1,2,522.39,7.28,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.0,130.0,221.282761,0.000000,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554..."
3,1,3,425.31,5.70,3,-77.315564,1.245136,-77.316070,1.244731,-77.315057,1.245540,0.0,130.0,221.282761,0.000000,"POLYGON ((-77.31506 1.24473, -77.31507 1.24554..."
4,1,4,425.31,5.70,4,-77.314566,1.245136,-77.315072,1.244731,-77.314059,1.245540,0.0,130.0,221.282761,0.000000,"POLYGON ((-77.31406 1.24473, -77.31407 1.24554..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,3,9995,1191.14,11.47,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,0.0,6.0,94.708580,0.000000,"POLYGON ((-77.28145 1.14848, -77.28146 1.14929..."
29996,3,9996,1283.66,12.42,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.0,6.0,94.708580,0.000000,"POLYGON ((-77.28045 1.14848, -77.28047 1.14929..."
29997,3,9997,1286.86,12.44,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,1.0,6.0,94.708580,15.784763,"POLYGON ((-77.27946 1.14848, -77.27947 1.14929..."
29998,3,9998,1287.95,12.45,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.0,6.0,94.708580,0.000000,"POLYGON ((-77.27846 1.14848, -77.27847 1.14929..."


In [11]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "origin_id": "hcf_uid",
    "pop": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "hcf_uid", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

In [12]:
pop_centroids_hcf

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_uid,duration_seconds,distance_km
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,NaN,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554...",1,518.30,7.24
1,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,NaN,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554...",1,521.19,7.27
2,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.000000,0.0,130.0,221.282761,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554...",1,522.39,7.28
3,3,-77.315564,1.245136,-77.316070,1.244731,-77.315057,1.245540,0.000000,0.0,130.0,221.282761,"POLYGON ((-77.31506 1.24473, -77.31507 1.24554...",1,425.31,5.70
4,4,-77.314566,1.245136,-77.315072,1.244731,-77.314059,1.245540,0.000000,0.0,130.0,221.282761,"POLYGON ((-77.31406 1.24473, -77.31407 1.24554...",1,425.31,5.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.28145 1.14848, -77.28146 1.14929...",3,1191.14,11.47
29996,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.28045 1.14848, -77.28047 1.14929...",3,1283.66,12.42
29997,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,15.784763,1.0,6.0,94.708580,"POLYGON ((-77.27946 1.14848, -77.27947 1.14929...",3,1286.86,12.44
29998,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.27846 1.14848, -77.27847 1.14929...",3,1287.95,12.45


Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [13]:
# For pasto, the HCF_ID was generated at the time of processing the OD Matrix. 
# Therefore, we need to read the file that contains those ids
healthcare_facilities_validated = gpd.read_file(data_temp + 'healthcare_facilities_emoc_hcfid.geojson')

healthcare_facilities_validated = healthcare_facilities_validated.rename(columns={
    "NombreSede": "facility_name",
    "project_validation": "Local_Validation"
})

In [14]:
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities_validated[['hcf_id','facility_name', 'longitude', 'latitude', 'Local_Validation']], 
                     left_on='hcf_uid', right_on='hcf_id', how='left')

In [15]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "longitude": "dest_lon",
    "latitude": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['hcf_uid'])

In [16]:
category_counts = healthcare_facilities_validated['Local_Validation'].value_counts()
print(category_counts)

Local_Validation
Private Basic EmOC            6
Private Comprehensive EmOC    3
Public Basic EmOC             2
Public Comprehensive EmOC     1
Name: count, dtype: int64


In [19]:
selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC', 
                       'Private Basic EmOC', 'Public Basic EmOC']

In [20]:
distances_duration_matrix = distances_duration_matrix[
    distances_duration_matrix['Local_Validation'].isin(selected_categories)]

distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,NaN,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554...",518.30,7.24,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
1,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,NaN,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554...",521.19,7.27,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
2,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.000000,0.0,130.0,221.282761,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554...",522.39,7.28,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
3,3,-77.315564,1.245136,-77.316070,1.244731,-77.315057,1.245540,0.000000,0.0,130.0,221.282761,"POLYGON ((-77.31506 1.24473, -77.31507 1.24554...",425.31,5.70,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
4,4,-77.314566,1.245136,-77.315072,1.244731,-77.314059,1.245540,0.000000,0.0,130.0,221.282761,"POLYGON ((-77.31406 1.24473, -77.31407 1.24554...",425.31,5.70,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.28145 1.14848, -77.28146 1.14929...",1191.14,11.47,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC
29996,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.28045 1.14848, -77.28047 1.14929...",1283.66,12.42,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC
29997,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,15.784763,1.0,6.0,94.708580,"POLYGON ((-77.27946 1.14848, -77.27947 1.14929...",1286.86,12.44,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC
29998,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.27846 1.14848, -77.27847 1.14929...",1287.95,12.45,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC


In [21]:
# creat subsets based on categories of 'Validation of HCFs Categorization'
categories = {
    "public_comprehensive_EmOC": ["Public Comprehensive EmOC"],
    "private_comprehensive_EmOC": ["Private Comprehensive EmOC"],
    "private_basic_EmOC": ["Private Basic EmOC"],
    "public_basic_EmOC": ["Public Basic EmOC"]
}

subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['Local_Validation'].str.contains('|'.join(values), na=False)
    ]
    for key, values in categories.items()
}

public_CEmOC = subsets["public_comprehensive_EmOC"]
private_CEmOC = subsets["private_comprehensive_EmOC"]
public_BEmOC = subsets["public_basic_EmOC"]
private_BEmOC = subsets["private_basic_EmOC"]

In [22]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)

In [23]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)
public_BEmOC_closest_3 = get_closest_3(public_BEmOC)
private_BEmOC_closest_3 = get_closest_3(private_BEmOC)

/var/folders/0s/jzqshn192vjfnj70wgny2bq00000gn/T/ipykernel_46100/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)
/var/folders/0s/jzqshn192vjfnj70wgny2bq00000gn/T/ipykernel_46100/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsm

In [24]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    public_CEmOC_closest_3, private_CEmOC_closest_3,
    public_BEmOC_closest_3, private_BEmOC_closest_3
])
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,NaN,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554...",518.30,7.24,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
1,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,NaN,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554...",632.89,8.40,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC
2,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,NaN,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554...",521.19,7.27,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
3,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,NaN,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554...",635.78,8.42,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC
4,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.000000,0.0,130.0,221.282761,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554...",522.39,7.28,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.28145 1.14848, -77.28146 1.14929...",1279.47,13.00,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC
9996,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.28045 1.14848, -77.28047 1.14929...",1371.99,13.95,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC
9997,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,15.784763,1.0,6.0,94.708580,"POLYGON ((-77.27946 1.14848, -77.27947 1.14929...",1375.19,13.97,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC
9998,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.000000,0.0,6.0,94.708580,"POLYGON ((-77.27846 1.14848, -77.27847 1.14929...",1376.28,13.98,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC


In [25]:
geometry = [Point(xy) for xy in zip(distances_duration_matrix['origin_lon'], distances_duration_matrix['origin_lat'])]
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry=geometry, crs="EPSG:4326")

In [26]:
gpkg_path = data_temp + 'distances_duration_3_closet_Emoc.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_3_closet_Emoc", driver="GPKG")

In [27]:
# Review and remove
origin_dest = distances_duration_matrix

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [28]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [29]:
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        0  -77.318557    1.245136      -77.319064        1.244731   
1        0  -77.318557    1.245136      -77.319064        1.244731   
2        1  -77.317559    1.245136      -77.318066        1.244731   
3        1  -77.317559    1.245136      -77.318066        1.244731   
4        2  -77.316561    1.245136      -77.317068        1.244731   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0      -77.318051         1.24554         NaN     3.0             28.0   
1      -77.318051         1.24554         NaN     3.0             28.0   
2      -77.317053         1.24554         NaN     0.0             28.0   
3      -77.317053         1.24554         NaN     0.0             28.0   
4      -77.316055         1.24554         0.0     0.0            130.0   

   pop_grid_pop                                           geometry  \
0           NaN  POLYGON ((-77.31805 1.24473, -77.31807 1.24554.

In [30]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [31]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [32]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [33]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [34]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,Weight,Pop_W
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,...,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554...",518.30,7.24,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC,0.032180,NaN
1,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,...,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554...",632.89,8.40,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC,0.005953,NaN
2,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,...,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554...",521.19,7.27,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC,0.030967,NaN
3,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,...,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554...",635.78,8.42,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC,0.005680,NaN
4,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.000000,0.0,130.0,...,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554...",522.39,7.28,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC,0.030475,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,0.000000,0.0,6.0,...,"POLYGON ((-77.28145 1.14848, -77.28146 1.14929...",1279.47,13.00,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0
9996,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.000000,0.0,6.0,...,"POLYGON ((-77.28045 1.14848, -77.28047 1.14929...",1371.99,13.95,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0
9997,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,15.784763,1.0,6.0,...,"POLYGON ((-77.27946 1.14848, -77.27947 1.14929...",1375.19,13.97,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0
9998,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.000000,0.0,6.0,...,"POLYGON ((-77.27846 1.14848, -77.27847 1.14929...",1376.28,13.98,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0


In [35]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()

In [36]:
origin_dest_sum

,hcf_id,Pop_W
0,1,25664.603030
1,2,19902.125964
2,3,21969.751214


In [37]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')

In [38]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_y
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,...,518.30,7.24,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC,0.032180,NaN,25664.603030
1,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,...,632.89,8.40,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC,0.005953,NaN,21969.751214
2,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,...,521.19,7.27,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC,0.030967,NaN,25664.603030
3,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,...,635.78,8.42,3,CRUZ ROJA COLOMBIANA SECCIONAL NARIÑO,-77.282908,1.213070,Private Comprehensive EmOC,0.005680,NaN,21969.751214
4,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.000000,0.0,130.0,...,522.39,7.28,1,HOSPITAL SAN RAFAEL DE PASTO,-77.287025,1.220111,Private Comprehensive EmOC,0.030475,0.0,25664.603030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,0.000000,0.0,6.0,...,1279.47,13.00,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964
29996,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.000000,0.0,6.0,...,1371.99,13.95,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964
29997,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,15.784763,1.0,6.0,...,1375.19,13.97,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964
29998,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.000000,0.0,6.0,...,1376.28,13.98,2,FUNDACION MARIA FORTALEZA,-77.284158,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964


In [39]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [40]:
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

In [41]:
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/0s/jzqshn192vjfnj70wgny2bq00000gn/T/ipykernel_46100/2370967217.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


In [42]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [43]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [44]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [45]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,...,1.220111,Private Comprehensive EmOC,0.032180,NaN,25664.603030,0.70,0.000027,8.777026e-07,1.980147e-06,2.839969e-02
1,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,NaN,3.0,28.0,...,1.213070,Private Comprehensive EmOC,0.005953,NaN,21969.751214,0.70,0.000032,1.896690e-07,1.980147e-06,2.839969e-02
2,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,...,1.220111,Private Comprehensive EmOC,0.030967,NaN,25664.603030,0.70,0.000027,8.446132e-07,1.906388e-06,2.734182e-02
3,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,NaN,0.0,28.0,...,1.213070,Private Comprehensive EmOC,0.005680,NaN,21969.751214,0.70,0.000032,1.809787e-07,1.906388e-06,2.734182e-02
4,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.000000,0.0,130.0,...,1.220111,Private Comprehensive EmOC,0.030475,0.0,25664.603030,0.70,0.000027,8.311906e-07,1.876491e-06,2.691303e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,0.000000,0.0,6.0,...,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964,0.35,0.000018,0.000000e+00,5.913691e-13,8.481540e-09
29996,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.000000,0.0,6.0,...,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964,0.35,0.000018,0.000000e+00,0.000000e+00,0.000000e+00
29997,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,15.784763,1.0,6.0,...,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964,0.35,0.000018,0.000000e+00,0.000000e+00,0.000000e+00
29998,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.000000,0.0,6.0,...,1.226692,Private Basic EmOC,0.000000,0.0,19902.125964,0.35,0.000018,0.000000e+00,0.000000e+00,0.000000e+00


In [46]:
max(origin_dest_acc.Accessibility_standard)

1.0

In [47]:
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [49]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

In [50]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [51]:
# Group by multiple columns and calculate the mean for numeric columns
# results_grid = results_grid.groupby(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard']).count().reset_index()
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])
type(results_grid)

geopandas.geodataframe.GeoDataFrame

In [52]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access', driver='GPKG')

In [53]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,2.839969e-02,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554..."
2,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,2.734182e-02,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554..."
4,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,2.691303e-02,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554..."
6,3,-77.315564,1.245136,-77.316070,1.244731,-77.315057,1.245540,8.645065e-02,"POLYGON ((-77.31506 1.24473, -77.31507 1.24554..."
8,4,-77.314566,1.245136,-77.315072,1.244731,-77.314059,1.245540,8.645065e-02,"POLYGON ((-77.31406 1.24473, -77.31407 1.24554..."
...,...,...,...,...,...,...,...,...,...
19990,9995,-77.281957,1.148884,-77.282462,1.148480,-77.281451,1.149289,8.481540e-09,"POLYGON ((-77.28145 1.14848, -77.28146 1.14929..."
19992,9996,-77.280959,1.148884,-77.281464,1.148480,-77.280453,1.149289,0.000000e+00,"POLYGON ((-77.28045 1.14848, -77.28047 1.14929..."
19994,9997,-77.279961,1.148884,-77.280467,1.148480,-77.279455,1.149289,0.000000e+00,"POLYGON ((-77.27946 1.14848, -77.27947 1.14929..."
19996,9998,-77.278963,1.148884,-77.279469,1.148480,-77.278457,1.149289,0.000000e+00,"POLYGON ((-77.27846 1.14848, -77.27847 1.14929..."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [58]:
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] > 0.000001, 'result'] = 2
results_grid.loc[results_grid['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.02, 'result'] = 0

In [56]:
category_counts = results_grid['result'].value_counts()
print(category_counts)

result
 2    4654
 0    3683
 1    1278
-1     385
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [59]:
results_grid['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid.loc[(results_grid['Accessibility_standard'] > 0.000001) & (results_grid['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.003) & (results_grid['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.019) & (results_grid['Accessibility_standard'] < 0.03), 'focused'] = 1

In [60]:
category_counts = results_grid['focused'].value_counts()
print(category_counts)

focused
0    8881
1    1119
Name: count, dtype: int64


In [61]:
results_grid = results_grid.loc[results_grid['result'] != -1]

In [62]:
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [63]:
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,result,focused
0,0,-77.318557,1.245136,-77.319064,1.244731,-77.318051,1.245540,0.028400,"POLYGON ((-77.31805 1.24473, -77.31807 1.24554...",0,1
2,1,-77.317559,1.245136,-77.318066,1.244731,-77.317053,1.245540,0.027342,"POLYGON ((-77.31705 1.24473, -77.31707 1.24554...",0,1
4,2,-77.316561,1.245136,-77.317068,1.244731,-77.316055,1.245540,0.026913,"POLYGON ((-77.31606 1.24473, -77.31607 1.24554...",0,1
6,3,-77.315564,1.245136,-77.316070,1.244731,-77.315057,1.245540,0.086451,"POLYGON ((-77.31506 1.24473, -77.31507 1.24554...",0,0
8,4,-77.314566,1.245136,-77.315072,1.244731,-77.314059,1.245540,0.086451,"POLYGON ((-77.31406 1.24473, -77.31407 1.24554...",0,0
...,...,...,...,...,...,...,...,...,...,...,...
19854,9927,-77.309911,1.149693,-77.310417,1.149289,-77.309406,1.150098,0.000016,"POLYGON ((-77.30941 1.14929, -77.30942 1.1501,...",2,0
19856,9928,-77.308913,1.149693,-77.309419,1.149289,-77.308408,1.150098,0.000020,"POLYGON ((-77.30841 1.14929, -77.30842 1.1501,...",2,0
19924,9962,-77.314887,1.148884,-77.315393,1.148480,-77.314382,1.149289,0.000004,"POLYGON ((-77.31438 1.14848, -77.3144 1.14929,...",2,0
19926,9963,-77.313889,1.148884,-77.314395,1.148480,-77.313384,1.149289,0.000011,"POLYGON ((-77.31338 1.14848, -77.3134 1.14929,...",2,0


In [64]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access-class', driver='GPKG')

In [1]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)

NameError: name 'results_grid' is not defined